In [ ]:
SELECT table_name, column_name
FROM spark_catalog.silver.information_schema.columns
WHERE lower(column_name) LIKE '%service%'
   OR lower(column_name) LIKE '%clienttype%'
   OR lower(column_name) LIKE '%tenancy%'
ORDER BY table_name, column_name;

In [ ]:
SELECT DISTINCT trim(clienttype) AS v
FROM <that_table>
WHERE clienttype IS NOT NULL AND trim(clienttype) <> ''
LIMIT 50;

In [ ]:
# Search columns across all tables in a database (Lakehouse schema)
db = "silver"
patterns = ["service", "clienttype", "tenancy"]

tables = [t.name for t in spark.catalog.listTables(db) if t.tableType.lower() != "view"]
hits = []

for tbl in tables:
    cols = [c.name.lower() for c in spark.table(f"{db}.{tbl}").schema.fields]
    if any(any(p in col for p in patterns) for col in cols):
        for c in cols:
            if any(p in c for p in patterns):
                hits.append((tbl, c))

display(spark.createDataFrame(hits, ["table_name", "column_name"]).orderBy("table_name", "column_name"))

--------------new------------------

In [ ]:
%sql
DROP TABLE IF EXISTS silver.silver_rdm_service_add;

CREATE TABLE silver.silver_rdm_service_add AS
WITH src AS (
    SELECT DISTINCT
        cp.z_src_system_instance                          AS service_src_sys_inst_id,
        trim(cp.cprod_service)                            AS service_src_name,
        lower(trim(cp.cprod_service))                     AS service_src_name_norm
    FROM silver.silver_care_product cp
    WHERE cp.cprod_service IS NOT NULL
      AND trim(cp.cprod_service) <> ''
      AND lower(trim(cp.cprod_service)) <> 'unknown'
),
src_with_id AS (
    SELECT
        service_src_sys_inst_id,
        service_src_name,
        concat(service_src_sys_inst_id, '|', service_src_name_norm) AS service_src_id_norm
    FROM src
),
rdm AS (
    SELECT DISTINCT
        lower(trim(service_src_id)) AS service_src_id_norm
    FROM silver.silver_rdm_service
    WHERE service_src_id IS NOT NULL
      AND trim(service_src_id) <> ''
)
SELECT
    -- 3 mandatory fields (per Mali)
    s.service_src_id_norm     AS service_src_id,
    s.service_src_name        AS service_src_name,
    s.service_src_sys_inst_id AS service_src_sys_inst_id,

    -- useful audit
    current_timestamp()       AS z_src_created_date_time,
    current_user()            AS z_src_created_by_user,
    current_timestamp()       AS z_src_modified_date_time,
    current_user()            AS z_src_modified_by_user,

    1                         AS z_order_is_active
FROM src_with_id s
LEFT JOIN rdm r
  ON s.service_src_id_norm = r.service_src_id_norm
WHERE r.service_src_id_norm IS NULL;

In [ ]:
%sql
SELECT count(*) AS new_rows FROM silver.silver_rdm_service_add;

%sql
SELECT service_src_sys_inst_id, count(*) AS c
FROM silver.silver_rdm_service_add
GROUP BY service_src_sys_inst_id
ORDER BY c DESC;

%sql
SELECT * FROM silver.silver_rdm_service_add LIMIT 50;

In [ ]:
आपण कोणते tables वापरले?
1) Source table (transactional-ish / curated)

silver.silver_care_product
इथून आपण “systems मध्ये दिसणारे services” (आपल्या case मध्ये cprod_service) काढले.

2) Reference table (SharePoint ingested RDM)

silver.silver_rdm_service
इथे आधीपासून जे service IDs SharePoint मध्ये आहेत ते आहेत, त्यामुळे “नवीन” काय आहे ते ठरवायला याचा वापर.

3) Output table

silver.silver_rdm_service_add
हा आपला “candidates/additions” टेबल, जो पुढे semantic model + Power Automate ने SharePoint मध्ये push करतो.

Code मध्ये CTE-by-CTE काय केलं?
CTE 1: src

उद्देश: source system मधून candidate services काढणे (duplicate कमी करणे)

From: silver.silver_care_product cp

Columns घेतले:

cp.z_src_system_instance → नाव दिलं: service_src_sys_inst_id
(हे “instance” आहे: IAPT 427, WIP001, SONE Y06008 वगैरे)

cp.cprod_service → trim करून नाव दिलं: service_src_name
(हे source system मधला service label)

cp.cprod_service → lower+trim करून नाव दिलं: service_src_name_norm
(matching साठी normalized version)

Filters:

cprod_service IS NOT NULL

trim(cprod_service) <> ''

lower(trim(cprod_service)) <> 'unknown'
(Unknown candidates push करायचे नाहीत म्हणून)

SELECT DISTINCT का?
एकाच service value हजारो rows मध्ये repeat असू शकते. त्यामुळे unique candidate set तयार केला.

CTE 2: src_with_id

उद्देश: Mali ने सांगितल्याप्रमाणे unique service_src_id बनवणे.

From: src

काय केलं:

service_src_id_norm = concat(service_src_sys_inst_id, '|', service_src_name_norm)

म्हणजे ID = instance + “|” + normalized name
उदा:

IAPT 427|intervention given

WIP001|helpline

SONE Y06008|rheumatology

हे का? (grain rule)
Mali म्हणतो: टेबलचा grain = instance by service.
त्यामुळे instance + name combine केलं की unique ID तयार होतो.

CTE 3: rdm

उद्देश: SharePoint/RDM मध्ये आधीपासून असलेले IDs काढणे, जेणेकरून duplicates टाळता येतील.

From: silver.silver_rdm_service

Columns घेतले:

service_src_id → lower+trim करून नाव दिलं: service_src_id_norm

Filter:

service_src_id IS NOT NULL

trim(service_src_id) <> ''

इथे आपण service_id (SharePoint generated) वापरला नाही, कारण तो source systems शी match होत नाही.

Final SELECT मध्ये काय केलं?
Join logic (मुख्य match)

Left join: src_with_id s LEFT JOIN rdm r

Join condition:

s.service_src_id_norm = r.service_src_id_norm

Why left join?
आपल्याला source मधल्या सर्व candidates ठेवायचे आहेत आणि जे RDM मध्ये आधीच आहेत ते drop करायचे आहेत.

Final filter:

WHERE r.service_src_id_norm IS NULL

याचा अर्थ:
RDM मध्ये match नाही → हा new service आहे → _add मध्ये टाका.

Output table मध्ये कोणते columns populate झाले?

silver.silver_rdm_service_add मध्ये आपण टाकले:

Mali ने सांगितलेले mandatory 3 fields:

service_src_id ← s.service_src_id_norm

service_src_name ← s.service_src_name

service_src_sys_inst_id ← s.service_src_sys_inst_id

Audit/ops fields:

z_src_created_date_time = current_timestamp()

z_src_created_by_user = current_user()

z_src_modified_date_time = current_timestamp()

z_src_modified_by_user = current_user()

z_order_is_active = 1

हे पुढे “who/when inserted” trace करायला useful आहे.

In [ ]:
---------------task new wip---------------------add tables

In [ ]:
%sql
DROP TABLE IF EXISTS silver.silver_rdm_service_add;

CREATE TABLE silver.silver_rdm_service_add AS
WITH src AS (
    -- Existing candidates from care_product
    SELECT DISTINCT
        cp.z_src_system_instance              AS service_src_sys_inst_id,
        trim(cp.cprod_service)                AS service_src_name,
        lower(trim(cp.cprod_service))         AS service_src_name_norm
    FROM silver.silver_care_product cp
    WHERE cp.cprod_service IS NOT NULL
      AND trim(cp.cprod_service) <> ''
      AND lower(trim(cp.cprod_service)) <> 'unknown'

    UNION ALL

    -- WIP rule (Monday): All WIP activity will be 'EAP'
    SELECT
        'WIP001' AS service_src_sys_inst_id,
        'EAP'    AS service_src_name,
        'eap'    AS service_src_name_norm
),
src_with_id AS (
    SELECT
        service_src_sys_inst_id,
        service_src_name,
        concat(service_src_sys_inst_id, '|', service_src_name_norm) AS service_src_id_norm
    FROM src
),
rdm AS (
    SELECT DISTINCT
        lower(trim(service_src_id)) AS service_src_id_norm
    FROM silver.silver_rdm_service
    WHERE service_src_id IS NOT NULL
      AND trim(service_src_id) <> ''
)
SELECT
    -- 3 mandatory fields (per Mali)
    s.service_src_id_norm        AS service_src_id,
    s.service_src_name           AS service_src_name,
    s.service_src_sys_inst_id    AS service_src_sys_inst_id,

    -- useful audit
    current_timestamp()          AS z_src_created_date_time,
    current_user()               AS z_src_created_by_user,
    current_timestamp()          AS z_src_modified_date_time,
    current_user()               AS z_src_modified_by_user,

    1                            AS z_order_is_active
FROM src_with_id s
LEFT JOIN rdm r
  ON s.service_src_id_norm = r.service_src_id_norm
WHERE r.service_src_id_norm IS NULL;

In [ ]:
%sql
SELECT *
FROM silver.silver_rdm_service_add
WHERE service_src_sys_inst_id = 'WIP001'
  AND lower(service_src_name) = 'eap';

Hi team, quick clarification needed on RDM Service – WIP logic for _add candidates.

Context

Monday spec for service_src_name says: “WIP: All WIP activity will be ‘EAP’”.

Mali also mentioned WIP can be hardcoded to EAP.

What I’m seeing

When generating silver_rdm_service_add, WIP instance WIP001 currently produces multiple candidate rows from silver_care_product.cprod_service:

EAP (hardcoded per spec)

Fees and Charges

Helpline

Interventions

Preventative Services
(IDs like WIP001|eap, WIP001|helpline, etc.)

Question
Should WIP in RDM Service be:

Strict: only a single service value EAP for WIP (so we should exclude other WIP-derived values from the source), OR

Flexible: allow additional WIP service values from source tables as candidates (and business will classify later)?

If option (1), please confirm whether we should filter out WIP rows from the generic candidate source and keep only the EAP hardcoded record for WIP001.

Thanks!

In [ ]:
---drj--------

In [ ]:
%sql
DROP TABLE IF EXISTS silver.silver_rdm_service_add;

CREATE TABLE silver.silver_rdm_service_add AS
WITH
-- dynamic WIP instances from source systems list
sys_wip AS (
    SELECT DISTINCT
        s.src_sys_inst_id AS service_src_sys_inst_id
    FROM silver.silver_rdm_source_systems s
    WHERE lower(trim(s.src_sys_id)) = 'wip'
),

-- generic candidates (as-is) from care_product
src_care_product AS (
    SELECT DISTINCT
        cp.z_src_system_instance                          AS service_src_sys_inst_id,
        trim(cp.cprod_service)                            AS service_src_name,
        concat(cp.z_src_system_instance,'|',lower(trim(cp.cprod_service))) AS service_src_id_norm
    FROM silver.silver_care_product cp
    WHERE cp.cprod_service IS NOT NULL
      AND trim(cp.cprod_service) <> ''
      AND lower(trim(cp.cprod_service)) <> 'unknown'
),

-- WIP rule: all WIP activity = EAP (for all WIP instances)
src_wip AS (
    SELECT DISTINCT
        w.service_src_sys_inst_id                         AS service_src_sys_inst_id,
        'EAP'                                             AS service_src_name,
        concat(w.service_src_sys_inst_id,'|','eap')        AS service_src_id_norm
    FROM sys_wip w
),

-- DRJ = MPB: tenancy-level ClientType -> service (instance fixed to MPB001)
-- Use ONE of the two blocks below depending on your column name in silver_drj_tenancies:

src_mpb AS (
    -- Variant A: column name = clienttype
    SELECT DISTINCT
        'MPB001'                                          AS service_src_sys_inst_id,
        trim(t.clienttype)                                AS service_src_name,
        concat('MPB001','|',lower(trim(t.clienttype)))     AS service_src_id_norm
    FROM silver.silver_drj_tenancies t
    WHERE t.clienttype IS NOT NULL
      AND trim(t.clienttype) <> ''

    /*
    -- Variant B: column name = client_type
    SELECT DISTINCT
        'MPB001'                                          AS service_src_sys_inst_id,
        trim(t.client_type)                               AS service_src_name,
        concat('MPB001','|',lower(trim(t.client_type)))    AS service_src_id_norm
    FROM silver.silver_drj_tenancies t
    WHERE t.client_type IS NOT NULL
      AND trim(t.client_type) <> ''
    */
),

src_all AS (
    SELECT service_src_sys_inst_id, service_src_name, service_src_id_norm FROM src_care_product
    UNION ALL
    SELECT service_src_sys_inst_id, service_src_name, service_src_id_norm FROM src_wip
    UNION ALL
    SELECT service_src_sys_inst_id, service_src_name, service_src_id_norm FROM src_mpb
),

rdm AS (
    SELECT DISTINCT
        lower(trim(service_src_id)) AS service_src_id_norm
    FROM silver.silver_rdm_service
    WHERE service_src_id IS NOT NULL
      AND trim(service_src_id) <> ''
)

SELECT DISTINCT
    s.service_src_id_norm        AS service_src_id,
    s.service_src_name           AS service_src_name,
    s.service_src_sys_inst_id    AS service_src_sys_inst_id,
    current_timestamp()          AS z_src_created_date_time,
    current_user()               AS z_src_created_by_user,
    current_timestamp()          AS z_src_modified_date_time,
    current_user()               AS z_src_modified_by_user,
    1                            AS z_order_is_active
FROM src_all s
LEFT JOIN rdm r
  ON lower(trim(s.service_src_id_norm)) = r.service_src_id_norm
WHERE r.service_src_id_norm IS NULL;

In [ ]:
%sql
DROP TABLE IF EXISTS silver.silver_rdm_service_add;

CREATE TABLE silver.silver_rdm_service_add AS
WITH
-- 1) generic candidates from care_product (as-is)
src_care_product AS (
    SELECT DISTINCT
        cp.z_src_system_instance AS service_src_sys_inst_id,
        trim(cp.cprod_service)   AS service_src_name,
        concat(cp.z_src_system_instance,'|',lower(trim(cp.cprod_service))) AS service_src_id_norm
    FROM silver.silver_care_product cp
    WHERE cp.cprod_service IS NOT NULL
      AND trim(cp.cprod_service) <> ''
      AND lower(trim(cp.cprod_service)) <> 'unknown'
),

-- 2) WIP rule (direct)
src_wip AS (
    SELECT
        'WIP001' AS service_src_sys_inst_id,
        'EAP'    AS service_src_name,
        'WIP001|eap' AS service_src_id_norm
),

-- 3) DRJ = MPB (direct MPB001) from tenancies client_type
src_mpb AS (
    SELECT DISTINCT
        'MPB001' AS service_src_sys_inst_id,
        trim(t.client_type) AS service_src_name,
        concat('MPB001','|',lower(trim(t.client_type))) AS service_src_id_norm
    FROM silver.silver_drj_tenancies t
    WHERE t.client_type IS NOT NULL
      AND trim(t.client_type) <> ''
),

-- 4) combine all
src_all AS (
    SELECT service_src_sys_inst_id, service_src_name, service_src_id_norm FROM src_care_product
    UNION ALL
    SELECT service_src_sys_inst_id, service_src_name, service_src_id_norm FROM src_wip
    UNION ALL
    SELECT service_src_sys_inst_id, service_src_name, service_src_id_norm FROM src_mpb
),

-- 5) existing RDM IDs
rdm AS (
    SELECT DISTINCT lower(trim(service_src_id)) AS service_src_id_norm
    FROM silver.silver_rdm_service
    WHERE service_src_id IS NOT NULL
      AND trim(service_src_id) <> ''
)

-- 6) only new rows
SELECT DISTINCT
    s.service_src_id_norm      AS service_src_id,
    s.service_src_name         AS service_src_name,
    s.service_src_sys_inst_id  AS service_src_sys_inst_id,
    current_timestamp()        AS z_src_created_date_time,
    current_user()             AS z_src_created_by_user,
    current_timestamp()        AS z_src_modified_date_time,
    current_user()             AS z_src_modified_by_user,
    1                          AS z_order_is_active
FROM src_all s
LEFT JOIN rdm r
  ON lower(trim(s.service_src_id_norm)) = r.service_src_id_norm
WHERE r.service_src_id_norm IS NULL;

In [ ]:
%sql
SELECT service_src_sys_inst_id, count(*) AS c
FROM silver.silver_rdm_service_add
GROUP BY service_src_sys_inst_id
ORDER BY c DESC;

In [ ]:
%sql
SELECT service_src_id, service_src_name, service_src_sys_inst_id
FROM silver.silver_rdm_service_add
WHERE service_src_sys_inst_id IN ('MPB001','WIP001')
ORDER BY service_src_sys_inst_id, service_src_name;

In [ ]:
DevOps 5201 (DRJ) update text (copy-paste)

What I did: Generated RDM Service ADD candidates as part of the shared “ALL systems” additions logic. For DRJ/MPB, derived service_src_name from tenancy-level client_type and created stable IDs as MPB001|<client_type>.
Evidence: Query filtered to service_src_sys_inst_id = MPB001 returns PRIVATE, REFERRER, SME.
Status: Ready for UAT.

In [ ]:
no care product

In [ ]:
%sql
DROP TABLE IF EXISTS silver.silver_rdm_service_add;

CREATE TABLE silver.silver_rdm_service_add AS
WITH
-- MPB / DRJ
src_mpb AS (
    SELECT DISTINCT
        concat('MPB001','|',lower(trim(t.client_type))) AS service_src_id_norm,
        trim(t.client_type) AS service_src_name,
        'MPB001' AS service_src_sys_inst_id
    FROM silver.silver_drj_tenancies t
    WHERE t.client_type IS NOT NULL
      AND trim(t.client_type) <> ''
),

-- WIP rule (single row)
src_wip AS (
    SELECT
        'WIP001|eap' AS service_src_id_norm,
        'EAP' AS service_src_name,
        'WIP001' AS service_src_sys_inst_id
),

src_all AS (
    SELECT * FROM src_mpb
    UNION ALL
    SELECT * FROM src_wip
),

rdm AS (
    SELECT DISTINCT lower(trim(service_src_id)) AS service_src_id_norm
    FROM silver.silver_rdm_service
    WHERE service_src_id IS NOT NULL
      AND trim(service_src_id) <> ''
)

SELECT DISTINCT
    s.service_src_id_norm     AS service_src_id,
    s.service_src_name        AS service_src_name,
    s.service_src_sys_inst_id AS service_src_sys_inst_id,
    current_timestamp()       AS z_src_created_date_time,
    current_user()            AS z_src_created_by_user,
    current_timestamp()       AS z_src_modified_date_time,
    current_user()            AS z_src_modified_by_user,
    1                         AS z_order_is_active
FROM src_all s
LEFT JOIN rdm r
  ON s.service_src_id_norm = r.service_src_id_norm
WHERE r.service_src_id_norm IS NULL;

In [ ]:
%sql
SELECT service_src_sys_inst_id, count(*) c
FROM silver.silver_rdm_service_add
GROUP BY service_src_sys_inst_id
ORDER BY c DESC;

In [ ]:
%sql
SELECT service_src_id, service_src_name, service_src_sys_inst_id
FROM silver.silver_rdm_service_add
WHERE service_src_sys_inst_id IN ('MPB001','WIP001')
ORDER BY service_src_sys_inst_id, service_src_name;

In [ ]:
%sql
SELECT service_src_id, count(*) c
FROM silver.silver_rdm_service_add
GROUP BY service_src_id
HAVING count(*) > 1;

In [ ]:
script

In [ ]:
neil mpb vs my

In [ ]:
Call Script (2–3 mins)

Hey, no worries. I can quickly walk you through what I did and how it compares to your approach.

1) What you did

From what I saw in your code, for MPB, you’re pulling the value from silver_drj_tenancies.client_type and building the candidate rows for the RDM Service Add logic.
You create a service_src_id using the MPB instance + the client type (so it becomes something like MPB001_<client_type>), set the service_src_name as the client type itself, and set service_src_sys_inst_id as MPB001.
Then you left join to silver_rdm_service and only keep rows that don’t already exist, so only new candidates get generated.

2) What I did

I did the same MVP logic, but kept it a bit simpler and more standardised for duplicates.
For MPB/DRJ, I also use silver_drj_tenancies.client_type as the source value.
I set service_src_sys_inst_id = MPB001, service_src_name = client_type, and I generate service_src_id as MPB001|lower(trim(client_type)) so that things like SME, sme, or SME don’t create multiple IDs.
For WIP, I followed the Monday rule that all WIP activity is EAP, so I generate the single candidate WIP001|eap.

Then, same as you, I left join to silver_rdm_service and only output rows where the ID is not already present.

3) What’s the same (MVP confirmation)

So yes, at MVP level our logic is the same:

same source for MPB (client_type)

same goal: generate only new candidates into the _add table

same existence check: left join to silver_rdm_service and keep only missing rows

4) What’s different

The only difference is the ID format and normalisation:

your MPB id is MPB001_<client_type>

mine is MPB001|<normalised client_type>

So the names/values are the same, but the ID strings can differ, which could matter if we want one consistent standard in SharePoint.

5) Quick question to align

Can we agree what the standard service_src_id format should be for MPB (underscore vs pipe, and whether we normalise case/spacing)?
Once we confirm that, I can align mine to match the team standard.

One-liner if he’s in a rush

“Both implementations do the same MVP thing: take MPB client_type and WIP EAP, create service IDs, then left-join to silver_rdm_service to output only new candidates. The only difference is the ID formatting and normalisation.”

In [ ]:
English Script (What I implemented)

So, I finished the MVP logic for RDM Service – Add table in the Create RDM SharePoint List Additions notebook.

Goal

The goal is to generate a candidate “add” table (silver.silver_rdm_service_add) that contains only new service values that are not already present in the main RDM Service table (silver.silver_rdm_service).
These candidates can then be pushed back to the SharePoint RDM Service list by Power Automate.

What I built

I drop and recreate the add table each run to keep it clean and repeatable.

I generate candidates for two systems based on the Monday/Mali requirements:

MPB/DRJ (same system)

Source: silver.silver_drj_tenancies

Business rule: use client_type as the service name

I create:

service_src_sys_inst_id = 'MPB001'

service_src_name = trim(client_type)

service_src_id = 'MPB001_' + lower(trim(client_type))

I exclude null/blank client_type values.

WIP

Business rule: “All WIP activity will be EAP”

So I generate a single fixed candidate row:

service_src_sys_inst_id = 'WIP001'

service_src_name = 'EAP'

service_src_id = 'WIP001_eap'

I union both candidate sets into one dataset.

To detect new values only, I normalise IDs using lower(trim()) and then:

Left join the candidates to silver.silver_rdm_service on the normalised service_src_id

Keep only rows where the RDM side is null (meaning the ID does not already exist)

I also add standard metadata columns:

created/modified timestamps and users (current_timestamp(), current_user())

z_order_is_active = 1 so all candidates are treated as active.

Output

The output is a small add table containing only the missing services, for example:

MPB001: SME / Private / Referrer (from client_type)

WIP001: EAP (single row)

So overall, this is the MVP “new candidates only” logic, ready for the next steps: adding to the semantic model and wiring the nightly Power Automate flow to insert these candidates into SharePoint.

In [ ]:
Step 1: MPB/DRJ block (monday rule: ClientType)
Source table

silver.silver_drj_tenancies t

आपण कोणता कॉलम वापरतो?

t.client_type (उदा SME / Private / Referrer)

आपण 3 mandatory fields तयार करतो (Mali ने सांगितलेले)

service_src_sys_inst_id

आपण constant देतो: 'MPB001'
कारण MVP मध्ये MPB एकच instance मानला आहे.

service_src_name

trim(t.client_type)
म्हणजे user-friendly name.

service_src_id_norm (हेच आपलं unique ID logic)

concat('MPB001','_', lower(trim(t.client_type)))
उदा:

MPB001_sme

MPB001_private

MPB001_referrer

Filter

client_type null/blank नसावा.

WHERE t.client_type IS NOT NULL
AND trim(t.client_type) <> ''

✅ त्यामुळे MPB साठी “service values” आपण tenancy मधून घेतो आणि ID stable ठेवतो (lower+trim).

Step 2: WIP block (monday rule: All WIP activity = EAP)
Source logic (single row)

WIP साठी Mali/Monday rule clear आहे:

“All WIP activity will be ‘EAP’”

म्हणून आपण 1 fixed candidate row बनवतो:

service_src_sys_inst_id = 'WIP001'

service_src_name = 'EAP'

service_src_id_norm = 'WIP001_eap'

✅ म्हणजे WIP साठी आपल्याला source tables scan करायची गरज नाही MVP मध्ये.

Step 3: Combine all candidates into one list
src_all AS (
  SELECT * FROM src_mpb
  UNION ALL
  SELECT * FROM src_wip
)

✅ म्हणजे आता आपल्याकडे “MPB candidates + WIP candidate” एकत्र झाले.

Step 4: Existing RDM Service IDs normalize करणे
Source table

silver.silver_rdm_service (SharePoint मधून आलेला main RDM table)

आपण existing IDs सुद्धा normalize करतो:

rdm AS (
  SELECT DISTINCT lower(trim(service_src_id)) AS service_src_id_norm
  FROM silver.silver_rdm_service
  WHERE service_src_id IS NOT NULL
    AND trim(service_src_id) <> ''
)

✅ कारण compare करताना spaces/case मुळे duplicates नको.

Step 5: New rows identify (core logic)

आता आपण src_all (candidates) ला rdm (existing) ला LEFT JOIN करतो:

FROM src_all s
LEFT JOIN rdm r
  ON s.service_src_id_norm = r.service_src_id_norm
WHERE r.service_src_id_norm IS NULL;

✅ अर्थ:

candidate ID जर RDM मध्ये already exists असेल → drop

candidate ID जर RDM मध्ये नसेल → हा नवीन → add table मध्ये ठेव

Step 6: Final output columns (add table मध्ये काय जातं?)

Final SELECT मध्ये आपण हे columns output देतो:

Mandatory 3 fields (सबसे important)

service_src_id → s.service_src_id_norm (underscore + lower)

service_src_name → s.service_src_name

service_src_sys_inst_id → s.service_src_sys_inst_id

Audit columns

z_src_created_date_time = current_timestamp()

z_src_created_by_user = current_user()

z_src_modified_date_time = current_timestamp()

z_src_modified_by_user = current_user()

Active flag

z_order_is_active = 1

✅ म्हणजे Power Automate ला SharePoint मध्ये insert करण्यासाठी सगळं basic metadata तयार.

Output expectation (तुझ्या screenshot प्रमाणे)

MPB001: 3 rows (SME / PRIVATE / REFERRER)

WIP001: 1 row (EAP)

हेच MVP साठी योग्य आहे.

In [ ]:
%sql
-- RDM SERVICE - ADD (candidates)
-- MERGED: CF + IAPT (Neil) + MPB/DRJ + WIP (Yo)
-- OUTPUT MUST BE ONLY 3 COLS:
--   service_src_id, service_src_name, service_src_sys_inst_id

CREATE OR REPLACE TABLE silver.silver_rdm_service_add AS
WITH
-- existing RDM service IDs (for "already seen" check)
rdm AS (
  SELECT DISTINCT lower(trim(service_src_id)) AS service_src_id_norm
  FROM silver.silver_rdm_service
  WHERE service_src_id IS NOT NULL AND trim(service_src_id) <> ''
),

-- Caseflow (CF) - per Neil
src_cf AS (
  SELECT DISTINCT
    trim(prod.id)   AS service_src_id,
    trim(prod.name) AS service_src_name,
    'CF001'         AS service_src_sys_inst_id
  FROM silver.silver_cf_tblcaseflow_contracts_product prod
  WHERE prod.id IS NOT NULL AND trim(prod.id) <> ''
),

-- IAPT - per Neil (ID = IAPT<service_id>_<treatment_type>, instance = IAPT<service_id>)
src_iapt AS (
  SELECT DISTINCT
    concat('IAPT', cast(treat.service_id as string), '_', trim(treat.treatment_type)) AS service_src_id,
    trim(treat.treatment_type)                                                     AS service_src_name,
    concat('IAPT', cast(treat.service_id as string))                                AS service_src_sys_inst_id
  FROM silver.silver_iapt_treatments treat
  WHERE treat.service_id IS NOT NULL
    AND treat.treatment_type IS NOT NULL
    AND trim(treat.treatment_type) <> ''
),

-- MPB / DRJ - ClientType from DRJ tenancies (MVP assumes MPB001)
src_mpb AS (
  SELECT DISTINCT
    concat('MPB001', '_', upper(trim(t.client_type))) AS service_src_id,
    upper(trim(t.client_type))                       AS service_src_name,
    'MPB001'                                         AS service_src_sys_inst_id
  FROM silver.silver_drj_tenancies t
  WHERE t.client_type IS NOT NULL
    AND trim(t.client_type) <> ''
),

-- WIP - per Monday definition: All WIP activity = EAP (single row)
src_wip AS (
  SELECT
    'WIP001_EAP' AS service_src_id,
    'EAP'        AS service_src_name,
    'WIP001'     AS service_src_sys_inst_id
),

-- union all candidate sources
src_all AS (
  SELECT * FROM src_cf
  UNION ALL
  SELECT * FROM src_iapt
  UNION ALL
  SELECT * FROM src_mpb
  UNION ALL
  SELECT * FROM src_wip
),

-- normalize for matching
src_all_norm AS (
  SELECT DISTINCT
    service_src_id,
    service_src_name,
    service_src_sys_inst_id,
    lower(trim(service_src_id)) AS service_src_id_norm
  FROM src_all
  WHERE service_src_id IS NOT NULL AND trim(service_src_id) <> ''
)

-- keep only brand-new IDs (not already in RDM service)
SELECT
  s.service_src_id,
  s.service_src_name,
  s.service_src_sys_inst_id
FROM src_all_norm s
LEFT JOIN rdm r
  ON s.service_src_id_norm = r.service_src_id_norm
WHERE r.service_src_id_norm IS NULL;

In [ ]:
%sql
SELECT service_src_sys_inst_id, count(*) c
FROM silver.silver_rdm_service_add
GROUP BY service_src_sys_inst_id
ORDER BY c DESC;

In [ ]:
%sql
SELECT *
FROM silver.silver_rdm_service_add
WHERE service_src_sys_inst_id IN ('MPB001','WIP001','CF001','IAPT319')
ORDER BY service_src_sys_inst_id, service_src_name;

I aligned the RDM Service ADD logic with the Monday spec and your feedback.
We now use CREATE OR REPLACE instead of DROP/CREATE for better stability.
I merged your Caseflow + IAPT extraction with my MPB/DRJ + WIP extraction into a single silver_rdm_service_add.
For each source we generate the same 3 mandatory columns: service_src_id, service_src_name, and service_src_sys_inst_id.
Then we filter only new values by left-joining to silver_rdm_service on a normalized service_src_id and keeping rows that don’t already exist.
I also removed the audit fields and active flag as requested, so the ADD table is now just the 3 fields needed for SharePoint ingestion.”

“For the IDs, I used underscore _ consistently (same as your pattern), e.g. MPB001_SME, WIP001_EAP, and IAPT319_Employment.

In [ ]:
%sql
CREATE OR REPLACE TABLE silver.silver_rdm_service_add AS
WITH
/* -----------------------------
   MPB / DRJ (Source: DRJ tenancies)
   Monday def: service_src_name = ClientType (tenancy level)
------------------------------ */
src_mpb AS (
  SELECT DISTINCT
    concat('MPB001', '_', upper(trim(t.client_type))) AS service_src_id,
    upper(trim(t.client_type))                       AS service_src_name,
    'MPB001'                                          AS service_src_sys_inst_id
  FROM silver.silver_drj_tenancies t
  WHERE t.client_type IS NOT NULL
    AND trim(t.client_type) <> ''
),

/* -----------------------------
   WIP (Monday def: all WIP activity = 'EAP')
   Single row only
------------------------------ */
src_wip AS (
  SELECT
    'WIP001_EAP' AS service_src_id,
    'EAP'        AS service_src_name,
    'WIP001'     AS service_src_sys_inst_id
),

/* -----------------------------
   Caseflow / CF (Neil)
   Source: silver_cf_tblcaseflow_contracts_product
   CF instance used here = CF001 (hardcoded)
------------------------------ */
src_cf AS (
  SELECT DISTINCT
    cast(prod.id as string)        AS service_src_id,
    trim(cast(prod.name as string)) AS service_src_name,
    'CF001'                         AS service_src_sys_inst_id
  FROM silver.silver_cf_tblcaseflow_contracts_product prod
  WHERE prod.id IS NOT NULL
    AND trim(cast(prod.id as string)) <> ''
    AND prod.name IS NOT NULL
    AND trim(cast(prod.name as string)) <> ''
),

/* -----------------------------
   IAPT (Neil)
   Source: silver_iapt_treatments
   service_src_sys_inst_id = IAPT + service_id (e.g. IAPT319)
   service_src_id = IAPT + service_id + '_' + treatment_type
------------------------------ */
src_iapt AS (
  SELECT DISTINCT
    concat('IAPT', cast(treat.service_id as string), '_', upper(trim(treat.treatment_type))) AS service_src_id,
    upper(trim(treat.treatment_type))                                                      AS service_src_name,
    concat('IAPT', cast(treat.service_id as string))                                        AS service_src_sys_inst_id
  FROM silver.silver_iapt_treatments treat
  WHERE treat.service_id IS NOT NULL
    AND treat.treatment_type IS NOT NULL
    AND trim(treat.treatment_type) <> ''
),

/* Combine all candidate rows */
src_all AS (
  SELECT * FROM src_cf
  UNION ALL SELECT * FROM src_iapt
  UNION ALL SELECT * FROM src_mpb
  UNION ALL SELECT * FROM src_wip
),

/* Normalize IDs for matching (avoid case/space issues) */
src_norm AS (
  SELECT DISTINCT
    s.service_src_id,
    s.service_src_name,
    s.service_src_sys_inst_id,
    lower(trim(cast(s.service_src_id as string))) AS service_src_id_norm
  FROM src_all s
  WHERE s.service_src_id IS NOT NULL
    AND trim(cast(s.service_src_id as string)) <> ''
),

rdm_norm AS (
  SELECT DISTINCT
    lower(trim(cast(service_src_id as string))) AS service_src_id_norm
  FROM silver.silver_rdm_service
  WHERE service_src_id IS NOT NULL
    AND trim(cast(service_src_id as string)) <> ''
)

/* Only NEW rows (not already in silver_rdm_service) */
SELECT
  s.service_src_id,
  s.service_src_name,
  s.service_src_sys_inst_id
FROM src_norm s
LEFT JOIN rdm_norm r
  ON s.service_src_id_norm = r.service_src_id_norm
WHERE r.service_src_id_norm IS NULL;

In [ ]:
select count(*) from silver.silver_rdm_service_add;

select service_src_sys_inst_id, count(*) from silver.silver_rdm_service_add group by 1 order by 2 desc;